# BM25 Retrieval Baseline

This notebook is the first reproducible retrieval baseline for Draftly. It indexes the canonical `data/processed/` legal corpus with SQLite FTS5 BM25, runs legal lookup queries, evaluates available rows from `evaluation/retrieval-gold.csv`, and saves machine-readable artifacts under `evaluation/runs/bm25-v1/`.

Scope is intentionally narrow: lexical BM25 only. There are no embeddings, LLM calls, graph expansion, authority-aware reranking, or temporal filtering in this baseline. Later experiments should compare those additions against these results.

The source PDF remains authoritative. Retrieved Markdown is a working representation and all generated case/statute links remain unverified until legal review.

## 1. Configuration

Set `DRAFTLY_RETRIEVAL_SMOKE=1` before execution to build a small disposable index for code verification. A normal notebook run indexes every non-case document plus the judgments marked as conveyancing matches.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    while current != current.parent:
        if (current / 'pyproject.toml').exists() and (current / 'data' / 'processed').exists():
            return current
        current = current.parent
    raise FileNotFoundError('Could not locate the Draftly repository root.')


REPO_ROOT = find_repo_root()
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
DOCUMENTS_CSV = PROCESSED_DIR / 'documents.csv'
CASES_JSONL = PROCESSED_DIR / 'cases.jsonl'
GOLD_CSV = REPO_ROOT / 'evaluation' / 'retrieval-gold.csv'

SMOKE = os.environ.get('DRAFTLY_RETRIEVAL_SMOKE', '0') == '1'
RUN_ID = 'bm25-v1-smoke' if SMOKE else 'bm25-v1'
RUN_DIR = REPO_ROOT / 'evaluation' / 'runs' / RUN_ID
INDEX_DIR = PROCESSED_DIR / 'retrieval-indexes'
INDEX_PATH = INDEX_DIR / f'{RUN_ID}.sqlite3'

CHUNK_WORDS = 320
CHUNK_OVERLAP_WORDS = 50
MIN_CHUNK_WORDS = 12
TOP_K = 10
EVAL_KS = (1, 5, 10)
EVAL_GOLD_STATUSES = {'draft', 'reviewed', 'lawyer_verified'}
FORCE_REBUILD = False

RUN_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository: {REPO_ROOT}')
print(f'Run: {RUN_ID}')
print(f'Index: {INDEX_PATH.relative_to(REPO_ROOT)}')

## 2. Select the canonical retrieval corpus

All statutes, amendments, gazettes, guides, and historical case-law volumes are included. Individual judgments are included only when `cases.jsonl` marks them as conveyancing matches. This keeps the baseline aligned with Draftly's first legal workflow while retaining the complete curated statutory layer.

In [ ]:
documents = pd.read_csv(DOCUMENTS_CSV, dtype=str, keep_default_na=False)
with CASES_JSONL.open(encoding='utf-8') as handle:
    cases = [json.loads(line) for line in handle if line.strip()]

conveyancing_case_ids = {
    row['case_id']
    for row in cases
    if str(row.get('conveyancing_match', '')).lower() in {'true', '1'}
}

is_individual_case = documents['kind'].eq('case-law')
selected_documents = documents[
    ~is_individual_case | documents['doc_id'].isin(conveyancing_case_ids)
].copy()
selected_documents = selected_documents[selected_documents['status'].eq('converted')]
selected_documents = selected_documents[selected_documents['text_path'].ne('')]

if SMOKE:
    non_cases = selected_documents[~selected_documents['kind'].eq('case-law')].head(40)
    case_docs = selected_documents[selected_documents['kind'].eq('case-law')].head(60)
    selected_documents = pd.concat([non_cases, case_docs], ignore_index=True)

missing_paths = []
for path_text in selected_documents['text_path']:
    if not (REPO_ROOT / Path(path_text)).is_file():
        missing_paths.append(path_text)
assert not missing_paths, f'Missing processed documents, first examples: {missing_paths[:5]}'

summary = (
    selected_documents.groupby('kind', dropna=False)
    .size()
    .rename('documents')
    .reset_index()
    .sort_values('documents', ascending=False)
)
display(summary)
print(f'Selected documents: {len(selected_documents):,}')
print(f'Conveyancing case IDs: {len(conveyancing_case_ids):,}')

## 3. Build a heading-aware SQLite FTS5 index

Markdown is split by headings and then into overlapping word windows. SQLite FTS5 provides a persistent BM25 index, allowing the full corpus to be searched without retaining all tokenized text in notebook memory. The index is rebuilt when the corpus manifest or baseline configuration changes.

In [ ]:
HEADING_RE = re.compile(r'^(#{1,6})\s+(.+?)\s*$')
SECTION_RE = re.compile(r'^(?:section\s+)?(\d+[A-Za-z]?(?:\s*\([^)]+\))*)\b', re.I)
TOKEN_RE = re.compile(r'[^\W_]+', re.UNICODE)


def corpus_fingerprint(frame: pd.DataFrame) -> str:
    digest = hashlib.sha256()
    digest.update(DOCUMENTS_CSV.read_bytes())
    digest.update(CASES_JSONL.read_bytes())
    digest.update('\n'.join(sorted(frame['doc_id'])).encode('utf-8'))
    digest.update(f'{CHUNK_WORDS}:{CHUNK_OVERLAP_WORDS}:{MIN_CHUNK_WORDS}'.encode())
    return digest.hexdigest()


def split_word_windows(text: str) -> list[str]:
    words = text.split()
    if len(words) < MIN_CHUNK_WORDS:
        return []
    if len(words) <= CHUNK_WORDS:
        return [' '.join(words)]
    step = CHUNK_WORDS - CHUNK_OVERLAP_WORDS
    windows = []
    for start in range(0, len(words), step):
        window = words[start:start + CHUNK_WORDS]
        if len(window) < MIN_CHUNK_WORDS:
            break
        windows.append(' '.join(window))
        if start + CHUNK_WORDS >= len(words):
            break
    return windows


def markdown_sections(text: str) -> list[tuple[str, str]]:
    sections = []
    heading = 'Document'
    body = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        match = HEADING_RE.match(line)
        if match:
            if body:
                sections.append((heading, ' '.join(body)))
            heading = match.group(2).strip()
            body = []
        elif line and not line.startswith('<!--'):
            body.append(line)
    if body:
        sections.append((heading, ' '.join(body)))
    return sections


def infer_section_id(source_id: str, kind: str, heading: str) -> str:
    if not source_id or kind not in {'statute', 'amendment'}:
        return ''
    match = SECTION_RE.match(heading.strip())
    if not match:
        return ''
    section = re.sub(r'\s+', '', match.group(1)).lower()
    return f'{source_id}-s{section}'


def iter_document_chunks(row: dict):
    text = (REPO_ROOT / Path(row['text_path'])).read_text(encoding='utf-8', errors='ignore')
    chunk_number = 0
    for heading, section_text in markdown_sections(text):
        for window in split_word_windows(section_text):
            chunk_number += 1
            yield {
                'chunk_id': f"{row['doc_id']}::c{chunk_number:04d}",
                'doc_id': row['doc_id'],
                'source_id': row.get('source_id', ''),
                'kind': row.get('kind', ''),
                'case_id': row['doc_id'] if row.get('kind') == 'case-law' else '',
                'section_id': infer_section_id(row.get('source_id', ''), row.get('kind', ''), heading),
                'title': row.get('title', ''),
                'citation': row.get('citation', ''),
                'heading': heading,
                'text': window,
                'text_path': row.get('text_path', ''),
            }


def index_fingerprint(path: Path) -> str | None:
    if not path.exists():
        return None
    try:
        with sqlite3.connect(path) as connection:
            row = connection.execute("SELECT value FROM index_meta WHERE key='fingerprint'").fetchone()
            return row[0] if row else None
    except sqlite3.DatabaseError:
        return None


def build_index(frame: pd.DataFrame, path: Path, fingerprint: str) -> dict:
    if path.exists():
        path.unlink()
    started = time.perf_counter()
    document_count = 0
    chunk_count = 0
    with sqlite3.connect(path) as connection:
        connection.execute('CREATE TABLE index_meta (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
        connection.execute(
            """CREATE VIRTUAL TABLE chunks USING fts5(
                chunk_id UNINDEXED, doc_id UNINDEXED, source_id UNINDEXED,
                kind UNINDEXED, case_id UNINDEXED, section_id UNINDEXED,
                title, citation, heading, text, text_path UNINDEXED,
                tokenize='unicode61 remove_diacritics 2'
            )"""
        )
        insert_sql = (
            'INSERT INTO chunks '
            '(chunk_id, doc_id, source_id, kind, case_id, section_id, title, citation, heading, text, text_path) '
            'VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)'
        )
        batch = []
        for row in frame.to_dict(orient='records'):
            document_count += 1
            for chunk in iter_document_chunks(row):
                batch.append(tuple(chunk.values()))
                chunk_count += 1
                if len(batch) >= 1000:
                    connection.executemany(insert_sql, batch)
                    batch.clear()
            if document_count % 500 == 0:
                print(f'Indexed {document_count:,} documents / {chunk_count:,} chunks')
        if batch:
            connection.executemany(insert_sql, batch)
        metadata = {
            'fingerprint': fingerprint,
            'documents': str(document_count),
            'chunks': str(chunk_count),
            'built_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        connection.executemany('INSERT INTO index_meta(key, value) VALUES (?, ?)', metadata.items())
        connection.commit()
    metadata['seconds'] = round(time.perf_counter() - started, 2)
    return metadata


fingerprint = corpus_fingerprint(selected_documents)
if FORCE_REBUILD or index_fingerprint(INDEX_PATH) != fingerprint:
    index_metadata = build_index(selected_documents, INDEX_PATH, fingerprint)
    print('Built index:', index_metadata)
else:
    with sqlite3.connect(INDEX_PATH) as connection:
        index_metadata = dict(connection.execute('SELECT key, value FROM index_meta').fetchall())
    print('Reusing compatible index:', index_metadata)

## 4. Query the baseline

FTS matches are ranked with BM25 using higher weights for titles, citations, and headings. Chunk results are collapsed to the best hit per document before top-k is returned, preventing a long judgment from filling the result list with duplicate passages.

In [ ]:
def fts_query(query: str) -> str:
    tokens = [token for token in TOKEN_RE.findall(query.casefold()) if len(token) > 1 or token.isdigit()]
    return ' OR '.join(f'"{token}"' for token in tokens)


def search_bm25(query: str, top_k: int = TOP_K) -> list[dict]:
    expression = fts_query(query)
    if not expression:
        return []
    candidate_k = max(top_k * 20, 100)
    sql = """
        SELECT chunk_id, doc_id, source_id, kind, case_id, section_id,
               title, citation, heading, text_path,
               bm25(chunks, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 2.0, 2.0, 1.0, 0.0) AS raw_score,
               snippet(chunks, 9, '[', ']', ' ... ', 28) AS snippet
        FROM chunks
        WHERE chunks MATCH ?
        ORDER BY raw_score
        LIMIT ?
    """
    with sqlite3.connect(INDEX_PATH) as connection:
        connection.row_factory = sqlite3.Row
        candidates = connection.execute(sql, (expression, candidate_k)).fetchall()

    hits = []
    seen_documents = set()
    for candidate in candidates:
        row = dict(candidate)
        if row['doc_id'] in seen_documents:
            continue
        seen_documents.add(row['doc_id'])
        row['rank'] = len(hits) + 1
        row['score'] = round(-float(row.pop('raw_score')), 6)
        hits.append(row)
        if len(hits) >= top_k:
            break
    return hits


def show_results(query: str, top_k: int = 5) -> pd.DataFrame:
    hits = search_bm25(query, top_k)
    print(f'Query: {query}')
    columns = ['rank', 'score', 'source_id', 'case_id', 'section_id', 'kind', 'title', 'citation', 'heading', 'snippet']
    frame = pd.DataFrame(hits)
    return frame.reindex(columns=columns)


sample_queries = [
    'requisites of a valid deed signed before a notary and witnesses',
    'registration of documents land registry certified copies',
    'partition action prescription co-owner possession',
]

for sample_query in sample_queries:
    display(show_results(sample_query, top_k=5))

## 5. Evaluate against `retrieval-gold.csv`

The evaluator never edits gold labels. It accepts draft, reviewed, and lawyer-verified rows but reports their status distribution. Authorities are matched through source IDs, section IDs, case IDs, and case citations. Metrics are macro-averaged across answerable questions.

When the gold file contains only its header, this section exits cleanly and explains that reviewed labels are required before scores can be claimed.

In [ ]:
def split_gold_values(value: str) -> list[str]:
    return [part.strip() for part in str(value or '').split(';') if part.strip()]


def normalize_identifier(value: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(value).casefold())


def expected_authorities(row: dict) -> list[dict]:
    fields = {
        'source': 'expected_source_ids',
        'section': 'expected_sections',
        'case': 'expected_case_ids',
        'citation': 'expected_case_citations',
    }
    expected = []
    for authority_type, field in fields.items():
        for value in split_gold_values(row.get(field, '')):
            expected.append({
                'key': f'{authority_type}:{normalize_identifier(value)}',
                'type': authority_type,
                'value': value,
                'normalized': normalize_identifier(value),
            })
    return expected


def hit_matches_expected(hit: dict, expected: dict) -> bool:
    target = expected['normalized']
    if not target:
        return False
    if expected['type'] == 'source':
        return target == normalize_identifier(hit.get('source_id', ''))
    if expected['type'] == 'case':
        return target in {normalize_identifier(hit.get('case_id', '')), normalize_identifier(hit.get('doc_id', ''))}
    if expected['type'] == 'citation':
        return target == normalize_identifier(hit.get('citation', ''))
    if expected['type'] == 'section':
        section_id = normalize_identifier(hit.get('section_id', ''))
        section_suffix = re.sub(r'^src\d+s', '', section_id)
        return target in {section_id, section_suffix}
    return False


def ndcg_at_k(relevant_ranks: list[int], n_relevant: int, k: int) -> float:
    dcg = sum(1.0 / math.log2(rank + 1) for rank in relevant_ranks if rank <= k)
    ideal = sum(1.0 / math.log2(rank + 1) for rank in range(1, min(n_relevant, k) + 1))
    return dcg / ideal if ideal else 0.0


def evaluate_gold(gold: pd.DataFrame, top_k: int = TOP_K):
    per_question = []
    prediction_rows = []
    for row in gold.to_dict(orient='records'):
        expected = expected_authorities(row)
        if not expected:
            continue
        hits = search_bm25(row['question_text'], top_k=top_k)
        found_rank = {}
        relevant_hit_ranks = []
        for hit in hits:
            matched = [item['key'] for item in expected if hit_matches_expected(hit, item)]
            if matched:
                relevant_hit_ranks.append(hit['rank'])
                for key in matched:
                    found_rank.setdefault(key, hit['rank'])
            prediction_rows.append({
                'question_id': row['question_id'],
                'question_text': row['question_text'],
                **{key: hit.get(key, '') for key in [
                    'rank', 'score', 'chunk_id', 'doc_id', 'source_id', 'case_id',
                    'section_id', 'kind', 'title', 'citation', 'heading', 'text_path', 'snippet'
                ]},
                'matched_gold_ids': ';'.join(matched),
            })
        gold_keys = {item['key'] for item in expected}
        first_rank = min(found_rank.values()) if found_rank else None
        metrics = {
            'question_id': row['question_id'],
            'gold_status': row.get('gold_status', ''),
            'n_expected': len(gold_keys),
            'n_found_at_10': sum(rank <= 10 for rank in found_rank.values()),
            'mrr': 1.0 / first_rank if first_rank else 0.0,
            'ndcg_at_10': ndcg_at_k(sorted(set(relevant_hit_ranks)), len(gold_keys), 10),
        }
        for k in EVAL_KS:
            found = sum(rank <= k for rank in found_rank.values())
            metrics[f'recall_at_{k}'] = found / len(gold_keys)
            metrics[f'all_recall_at_{k}'] = float(found == len(gold_keys))
        per_question.append(metrics)

    per_question_frame = pd.DataFrame(per_question)
    predictions_frame = pd.DataFrame(prediction_rows)
    if per_question_frame.empty:
        return {}, per_question_frame, predictions_frame
    metric_columns = [
        *(f'recall_at_{k}' for k in EVAL_KS),
        *(f'all_recall_at_{k}' for k in EVAL_KS),
        'mrr', 'ndcg_at_10',
    ]
    aggregate = {column: round(float(per_question_frame[column].mean()), 6) for column in metric_columns}
    aggregate['questions_scored'] = int(len(per_question_frame))
    return aggregate, per_question_frame, predictions_frame


gold = pd.read_csv(GOLD_CSV, dtype=str, keep_default_na=False)
gold = gold[gold['gold_status'].isin(EVAL_GOLD_STATUSES)].copy()
gold = gold[gold['question_text'].str.strip().ne('')]

run_config = {
    'run_id': RUN_ID,
    'retriever': 'sqlite-fts5-bm25',
    'corpus_fingerprint': fingerprint,
    'documents': int(len(selected_documents)),
    'chunks': int(index_metadata.get('chunks', 0)),
    'chunk_words': CHUNK_WORDS,
    'chunk_overlap_words': CHUNK_OVERLAP_WORDS,
    'top_k': TOP_K,
    'gold_statuses': sorted(EVAL_GOLD_STATUSES),
    'sqlite_version': sqlite3.sqlite_version,
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'smoke': SMOKE,
}
(RUN_DIR / 'config.json').write_text(json.dumps(run_config, indent=2) + '\n', encoding='utf-8')

if gold.empty:
    print('No scoreable gold rows are available yet.')
    print('Add draft/reviewed/lawyer_verified rows to evaluation/retrieval-gold.csv, then rerun this section.')
    aggregate_metrics = {}
else:
    print('Gold status counts:')
    display(gold['gold_status'].value_counts().rename_axis('status').to_frame('questions'))
    aggregate_metrics, per_question_metrics, predictions = evaluate_gold(gold, top_k=TOP_K)
    per_question_metrics.to_csv(RUN_DIR / 'per-question.csv', index=False, quoting=csv.QUOTE_MINIMAL)
    predictions.to_csv(RUN_DIR / 'predictions.csv', index=False, quoting=csv.QUOTE_MINIMAL)
    (RUN_DIR / 'metrics.json').write_text(json.dumps(aggregate_metrics, indent=2) + '\n', encoding='utf-8')
    display(pd.DataFrame([aggregate_metrics]))
    display(per_question_metrics.sort_values(['recall_at_10', 'mrr']).head(10))

print(f'Run artifacts: {RUN_DIR.relative_to(REPO_ROOT)}')

## 6. Baseline interpretation

This run is the lexical lower bound for the retrieval plan. Record its metrics before adding any component. Future experiments should change one stage at a time and preserve the same gold questions and corpus snapshot:

1. BM25 plus curated topic routing.
2. BM25 plus graph expansion.
3. Authority-aware reranking.
4. Temporal filtering.
5. Dense retrieval only if the preceding ablations reveal lexical recall gaps.

Do not claim legal correctness from draft gold labels or unverified citation links.